# Notebook 04 — Debit Prediction (FR5)

Predicts next debit date and amount for each recurring subscription.
Compares Rule-Based vs LR (gap-based) vs ARIMA.
Outputs `debit_predictions.csv`.

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression

try:
    from statsmodels.tsa.arima.model import ARIMA
    ARIMA_AVAILABLE = True
    print('✅ ARIMA available')
except ImportError:
    ARIMA_AVAILABLE = False
    print('⚠️  statsmodels not installed — ARIMA disabled. Run: pip install statsmodels')

print('✅ Imports done')


✅ ARIMA available
✅ Imports done


## 1. Load Data

In [4]:
DATA_PATH = r"C:\Users\mansi.apet\Downloads\AI_SUBSCRIPTION_SYSTEM_FINAL\data\transactions_patterns.csv"

df      = pd.read_csv(DATA_PATH, parse_dates=['Date'])
df_rec  = df[df['Is_Recurring'] == 1].copy()

print(f'Total rows    : {len(df):,}')
print(f'Recurring rows: {len(df_rec):,}')
df_rec.head()


Total rows    : 137,700
Recurring rows: 74,582


,CustomerID,CustomerName,TransactionID,Date,Description,Amount,Balance,Merchant,TransactionType,SubscriptionFlag,Status,Frequency,Description_Clean,NLP_Sub_Pred,NLP_Sub_Prob,Is_Recurring,Inferred_Freq,Occurrence_Count
1,CUST100000,Karan Verma,TXN31429110,2023-01-14,NETFLIX.COM MONTHLY SUB,661.05,223750.99,Netflix,DEBIT,1,SUCCESS,monthly,NETFLIX.COM MONTHLY SUBSCRIPTION,1,0.9974,1,Monthly,18
3,CUST100000,Karan Verma,TXN55176955,2023-02-14,NETFLIX.COM MONTHLY SUB,654.12,2667703.75,Netflix,DEBIT,1,SUCCESS,monthly,NETFLIX.COM MONTHLY SUBSCRIPTION,1,0.9974,1,Monthly,18
6,CUST100000,Karan Verma,TXN45503389,2023-03-14,NETFLIX.COM MONTHLY SUB,631.14,2667072.61,Netflix,DEBIT,1,SUCCESS,monthly,NETFLIX.COM MONTHLY SUBSCRIPTION,1,0.9974,1,Monthly,18
9,CUST100000,Karan Verma,TXN60806024,2023-04-14,NETFLIX.COM MONTHLY SUB,631.14,2666441.47,Netflix,DEBIT,1,SUCCESS,monthly,NETFLIX.COM MONTHLY SUBSCRIPTION,1,0.9974,1,Monthly,18
11,CUST100000,Karan Verma,TXN58537831,2023-05-14,NETFLIX.COM MONTHLY SUB,626.06,2665815.41,Netflix,DEBIT,1,SUCCESS,monthly,NETFLIX.COM MONTHLY SUBSCRIPTION,1,0.9974,1,Monthly,18


## 2. Prediction Functions

In [8]:
def rule_next_date(last_date, median_gap):
    """Rule-based: last date + median gap."""
    return last_date + pd.Timedelta(days=int(median_gap))


def lr_next_date(last_date, gaps):
    """
    LR (gap-based): fits a line through gap values to predict the next gap.
    This avoids encoding ordinal dates directly and is more stable.
    """
    if len(gaps) < 2:
        return last_date + pd.Timedelta(days=int(np.median(gaps)))
    X   = np.arange(len(gaps)).reshape(-1, 1)
    lr  = LinearRegression().fit(X, gaps)
    gap = max(1, int(lr.predict([[len(gaps)]])[0]))
    return last_date + pd.Timedelta(days=gap)


def arima_next_amount(amounts):
    """ARIMA(1,1,1) on amount series. Only used when variance > 0."""
    if not ARIMA_AVAILABLE or len(amounts) < 10 or np.std(amounts) < 0.1:
        return None    # Signal: fall back to mean
    try:
        m = ARIMA(amounts, order=(1,1,1)).fit()
        return round(float(m.forecast(1)[0]), 2)
    except:
        return None


## 3. Generate Predictions for All Subscriptions

In [7]:
def predict_debit(df):
    """
    Predicts next debit date + amount for each recurring (CustomerID, Merchant) pair.
    Keeps Rule-Based and LR columns for comparison.
    Final prediction uses Rule-Based (most reliable for fixed-schedule subscriptions).
    """
    rows, skipped = [], []

    for (cust, merchant), group in df[df['Is_Recurring']==1].groupby(['CustomerID','Merchant']):
        g = group.drop_duplicates('Date').sort_values('Date')

        if len(g) < 3:
            skipped.append({'reason':'insufficient_data','cust':cust,'merchant':merchant})
            continue

        gaps = g['Date'].diff().dt.days.dropna()
        gaps = gaps[gaps > 0]

        if len(gaps) == 0:
            skipped.append({'reason':'no_valid_gaps','cust':cust,'merchant':merchant})
            continue

        last_date   = g['Date'].iloc[-1]
        amounts     = g['Amount'].tolist()
        mean_amount = round(float(np.mean(amounts)), 2)

        r_date  = rule_next_date(last_date, gaps.median())
        l_date  = lr_next_date(last_date, gaps.values)
        a_amt   = arima_next_amount(amounts)   # None if not applicable

        rows.append({
            'CustomerID':         cust,
            'Merchant':           merchant,
            'num_transactions':   len(g),
            'last_date':          last_date,
            'median_gap':         int(gaps.median()),
            'gap_std':            round(float(gaps.std()), 2),
            'confidence':         'HIGH' if gaps.std() < 3 else ('MEDIUM' if gaps.std() < 7 else 'LOW'),
            'rule_next_date':     r_date,
            'lr_next_date':       l_date,
            'rule_next_amount':   mean_amount,
            'lr_next_amount':     mean_amount,
            'arima_next_amount':  a_amt if a_amt is not None else mean_amount,
            'arima_used':         a_amt is not None,
            # Final decision (most reliable for fixed-schedule subscriptions)
            'final_next_date':    r_date,
            'final_next_amount':  mean_amount,
        })

    result_df = pd.DataFrame(rows)
    print(f'Predictions generated : {len(result_df):,}')
    print(f'Skipped               : {len(skipped):,}')
    return result_df


result_df = predict_debit(df)
result_df.head()


c:\Users\mansi.apet\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\mansi.apet\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\mansi.apet\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\mansi.apet\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maxi

Predictions generated : 3,066
Skipped               : 1


,CustomerID,Merchant,num_transactions,last_date,median_gap,gap_std,confidence,rule_next_date,lr_next_date,rule_next_amount,lr_next_amount,arima_next_amount,arima_used,final_next_date,final_next_amount
0,CUST100000,Netflix,18,2024-06-14,31,0.87,HIGH,2024-07-15,2024-07-14,645.62,645.62,644.85,True,2024-07-15,645.62
1,CUST100001,Amazon,18,2024-06-22,31,0.87,HIGH,2024-07-23,2024-07-22,299.62,299.62,299.42,True,2024-07-23,299.62
2,CUST100001,Coursera,18,2024-06-18,31,0.87,HIGH,2024-07-19,2024-07-18,3987.15,3987.15,3966.86,True,2024-07-19,3987.15
3,CUST100002,AnytimeFitness,77,2024-06-24,7,0.80,HIGH,2024-07-01,2024-06-30,249.16,249.16,249.16,True,2024-07-01,249.16
4,CUST100002,Google,17,2024-06-04,31,7.70,LOW,2024-07-05,2024-07-06,129.59,129.59,129.13,True,2024-07-05,129.59


## 4. Model Evaluation (Leave-One-Out)

In [ ]:

def evaluate_models(df):
    """Leave-one-out: train on all rows except last, predict last."""
    rows = []
    for (cust, merchant), group in df[df['Is_Recurring']==1].groupby(['CustomerID','Merchant']):
        g = group.drop_duplicates('Date').sort_values('Date')
        if len(g) < 10: continue

        train, actual_row = g.iloc[:-1], g.iloc[-1]
        gaps = train['Date'].diff().dt.days.dropna()
        gaps = gaps[gaps > 0]
        if len(gaps) == 0: continue

        actual_date = actual_row['Date']
        actual_amt  = actual_row['Amount']

        r_date = rule_next_date(train['Date'].iloc[-1], gaps.median())
        l_date = lr_next_date(train['Date'].iloc[-1], gaps.values)

        mean_amt  = float(train['Amount'].mean())
        arima_amt = arima_next_amount(train['Amount'].tolist())

        rows.append({
            'rule_date_err':  abs((r_date - actual_date).days),
            'lr_date_err':    abs((l_date - actual_date).days),
            'rule_amt_err':   abs(mean_amt  - actual_amt),
            'arima_amt_err':  abs(arima_amt - actual_amt) if arima_amt else None,
            'arima_used':     arima_amt is not None,
        })

    return pd.DataFrame(rows)


eval_df = evaluate_models(df)

print(f'Evaluated on {len(eval_df):,} subscription groups\n')
print('DATE PREDICTION (mean absolute error in days):')
print(f'  Rule-Based  : {eval_df["rule_date_err"].mean():.2f} days')
print(f'  LR Gap-Based: {eval_df["lr_date_err"].mean():.2f} days')
print()
print('AMOUNT PREDICTION (mean absolute error):')
print(f'  Rule-Based  : {eval_df["rule_amt_err"].mean():.2f}')
if ARIMA_AVAILABLE:
    n_arima = eval_df['arima_used'].sum()
    print(f'  ARIMA fits  : {n_arima} groups')
    if n_arima > 0:
        print(f'  ARIMA error : {eval_df.loc[eval_df["arima_used"],"arima_amt_err"].mean():.2f}')
    else:
        print('  ARIMA note  : Subscription amounts are constant → ARIMA adds no value')
        print('                Rule-based mean is the correct approach for fixed amounts')


c:\Users\mansi.apet\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\mansi.apet\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\mansi.apet\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\mansi.apet\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maxi

Evaluated on 3,066 subscription groups

DATE PREDICTION (mean absolute error in days):
  Rule-Based  : 0.28 days
  LR Gap-Based: 1.42 days

AMOUNT PREDICTION (mean absolute error):
  Rule-Based  : 16.11
  ARIMA fits  : 3066 groups
  ARIMA error : 17.30


## 5. ARIMA Demo (Netflix)

In [12]:
def arima_demo(df):
    if not ARIMA_AVAILABLE:
        print('statsmodels not installed. Run: pip install statsmodels')
        return

    g = df[df['Is_Recurring']==1]
    g = g[g['Merchant']=='Netflix'].drop_duplicates('Date').sort_values('Date')

    if len(g) < 10:
        print('Not enough Netflix data for ARIMA demo.')
        return

    train, actual = g['Amount'].values[:-1], g['Amount'].values[-1]
    pred = arima_next_amount(list(train))

    print('=== ARIMA Demo — Netflix ===')
    if pred is None:
        print(f'Amounts are constant (₹{train[0]:.2f}) — ARIMA not applicable.')
        print(f'Using mean: ₹{np.mean(train):.2f}  |  Actual: ₹{actual:.2f}')
    else:
        print(f'Predicted : ₹{pred:.2f}  |  Actual: ₹{actual:.2f}  |  Error: ₹{abs(pred-actual):.2f}')

arima_demo(df)


=== ARIMA Demo — Netflix ===
Predicted : ₹649.50  |  Actual: ₹627.02  |  Error: ₹22.48
